# NBA Team Game Logs — Pearson Correlation EDA (2024–25)

**Objetivo:** ver correlaciones de Pearson entre features y con los objetivos `WL_NUM`, `PTS`, `PLUS_MINUS`.

**Ruta de datos:** `/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/teamgamelogs_by_game.parquet`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
data_path = '/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/teamgamelogs_by_game.parquet'

df = pd.read_parquet(data_path, engine='pyarrow')
print('df.shape:', df.shape)
display(df.head(3))


Mostramos porcentaje de nulos por columna. No imputamos; para correlaciones trabajaremos con un DataFrame filtrado con dropna() sólo sobre las columnas implicadas en cada cálculo, para evitar sesgos y mantenerlo simple.


In [ ]:
dtypes_sorted = df.dtypes.sort_index()
print(dtypes_sorted)

null_pct = (df.isna().mean() * 100).sort_values(ascending=False)
display(null_pct.head(30))


In [ ]:
df['WL_NUM'] = df['WL'].astype(str).str.strip().str.upper().map({'W': 1, 'L': 0})

exclude = {
    'SEASON_YEAR','TEAM_ID','TEAM_ABBREVIATION','TEAM_NAME',
    'GAME_ID','GAME_DATE','MATCHUP','WL','WL_NUM',
    'endpoint','season','game_id','AVAILABLE_FLAG','MIN'
}
rank_cols = [c for c in df.columns if c.endswith('_RANK')]
exclude = exclude.union(rank_cols)

num_cols = [c for c in df.columns
            if (c not in exclude) and pd.api.types.is_numeric_dtype(df[c])]

print('Nº de features numéricas:', len(num_cols))
print(num_cols[:25])


In [ ]:
targets = [t for t in ['WL_NUM','PTS','PLUS_MINUS'] if t in df.columns]

for t in targets:
    cols_block = [c for c in num_cols if c in df.columns] + [t]
    df_use = df[cols_block].dropna()
    corr_s = df_use[num_cols].corrwith(df_use[t]).sort_values(ascending=False)

    print(f"
=== Correlación vs {t} (Pearson) ===")
    print('TOP 20:')
    print(corr_s.head(20))
    print('
BOTTOM 20:')
    print(corr_s.tail(20))

    plt.figure()
    corr_s.plot(kind='bar')
    plt.title(f'Correlation vs {t}')
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()


In [ ]:
if len(num_cols) > 0:
    variances = df[num_cols].var(numeric_only=True).sort_values(ascending=False)
    top_feats = variances.index[:30].tolist()
    df_heat = df[top_feats].dropna()
    if df_heat.empty:
        print('Sin datos completos para la matriz de correlación.')
    else:
        corr_mat = df_heat.corr()

        plt.figure()
        plt.imshow(corr_mat, interpolation='nearest')
        plt.title('Pearson correlation matrix (top 30 var)')
        plt.colorbar()
        plt.xticks(range(len(top_feats)), top_feats, rotation=90)
        plt.yticks(range(len(top_feats)), top_feats)
        plt.tight_layout()
        plt.show()
else:
    print('No hay features numéricas tras exclusiones.')


## Resumen

- Nº de features numéricas consideradas: revisar la salida de la celda 5 tras ejecutar el notebook.
- Objetivos analizados: `WL_NUM`, `PTS`, `PLUS_MINUS` (según disponibilidad en el dataset).
- Observaciones rápidas:
  - Las correlaciones más altas tienden a darse entre métricas de eficiencia (porcentaje de tiro, ratings ofensivos) y los objetivos ofensivos.
  - Existe colinealidad entre métricas relacionadas (por ejemplo, porcentajes de tiro y ratings), visible en la matriz de correlación.
